In [1]:
import inspect

from dataclasses import dataclass

from plasmapy import formulary


@dataclass
class blob_of_plasma:
    n_e: u.Quantity
    T_e: u.Quantity
    B: u.Quantity


blob = blob_of_plasma(100 * u.m ** -3, 200 * u.eV, 1 * u.T)

blob_of_plasma(n_e=<Quantity 100. 1 / m3>, T_e=<Quantity 200. eV>, B=<Quantity 1. T>)

In [2]:
formulary.Bohm_diffusion

<function plasmapy.formulary.parameters.Bohm_diffusion(T_e: Unit("K"), B: Unit("T")) -> Unit("m2 / s")>

In [3]:
fields_of_blob = dict(inspect.getmembers(blob))

{'B': <Quantity 1. T>,
 'T_e': <Quantity 200. eV>,
 '__annotations__': {'n_e': astropy.units.quantity.Quantity,
  'T_e': astropy.units.quantity.Quantity,
  'B': astropy.units.quantity.Quantity},
 '__class__': __main__.blob_of_plasma,
 '__dataclass_fields__': {'n_e': Field(name='n_e',type=<class 'astropy.units.quantity.Quantity'>,default=<dataclasses._MISSING_TYPE object at 0x7fa181c04fd0>,default_factory=<dataclasses._MISSING_TYPE object at 0x7fa181c04fd0>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),_field_type=_FIELD),
  'T_e': Field(name='T_e',type=<class 'astropy.units.quantity.Quantity'>,default=<dataclasses._MISSING_TYPE object at 0x7fa181c04fd0>,default_factory=<dataclasses._MISSING_TYPE object at 0x7fa181c04fd0>,init=True,repr=True,hash=None,compare=True,metadata=mappingproxy({}),_field_type=_FIELD),
  'B': Field(name='B',type=<class 'astropy.units.quantity.Quantity'>,default=<dataclasses._MISSING_TYPE object at 0x7fa181c04fd0>,default_factory=<dataclass

In [4]:
sig = inspect.signature(formulary.Bohm_diffusion)

<Signature (T_e: Unit("K"), B: Unit("T")) -> Unit("m2 / s")>

In [5]:
sig.parameters

mappingproxy({'T_e': <Parameter "T_e: Unit("K")">,
              'B': <Parameter "B: Unit("T")">})

In [6]:
filtered_fields_of_blob = {name: fields_of_blob[name] for name in sig.parameters}

{'T_e': <Quantity 200. eV>, 'B': <Quantity 1. T>}

In [7]:
bound = sig.bind_partial(**filtered_fields_of_blob)

<BoundArguments (T_e=<Quantity 200. eV>, B=<Quantity 1. T>)>

In [8]:
bound.args, bound.kwargs

((<Quantity 200. eV>, <Quantity 1. T>), {})

In [9]:
formulary.Bohm_diffusion(*bound.args, **bound.kwargs)

<Quantity 12.5 m2 / s>

In [10]:
import functools

partial_Bohm_diffusion = functools.partial(
    formulary.Bohm_diffusion, *bound.args, **bound.kwargs
)

functools.partial(<function Bohm_diffusion at 0x7fa103595940>, <Quantity 200. eV>, <Quantity 1. T>)

In [11]:
partial_Bohm_diffusion()

<Quantity 12.5 m2 / s>

In [12]:
partialmethod_Bohm_diffusion = functools.partialmethod(
    formulary.Bohm_diffusion, *bound.args, **bound.kwargs
)

functools.partialmethod(<function Bohm_diffusion at 0x7fa103595940>, <Quantity 200. eV>, <Quantity 1. T>, )

In [13]:
formulary.Bohm_diffusion.__name__

'Bohm_diffusion'

In [14]:
setattr(blob_of_plasma, formulary.Bohm_diffusion.__name__, partial_Bohm_diffusion)

In [15]:
blob.Bohm_diffusion()

<Quantity 12.5 m2 / s>

In [16]:
blob.Bohm_diffusion?

Signature:      blob.Bohm_diffusion() -> Unit("m2 / s")
Call signature: blob.Bohm_diffusion(*args, **kwargs)
Type:           partial
String form:    functools.partial(<function Bohm_diffusion at 0x7fa103595940>, <Quantity 200. eV>, <Quantity 1. T>)
File:           /usr/lib/python3.9/functools.py
Docstring:     
partial(func, *args, **keywords) - new function with partial application
of the given arguments and keywords.


## Summarizing

In [17]:
sig.replace()

<Signature (T_e: Unit("K"), B: Unit("T")) -> Unit("m2 / s")>

In [45]:
def get_partial_formulary_function(target_obj, function):
    fields_of_blob = dict(inspect.getmembers(target_obj))
    sig = inspect.signature(function)
    filtered_fields_of_blob = {
        name: fields_of_blob[name] for name in sig.parameters if name in fields_of_blob
    }
    bound = sig.bind_partial(**filtered_fields_of_blob)
    wrapper = functools.partial(function, *bound.args, **bound.kwargs)
    #     wrapper.__doc__ = function.__doc__
    wrapper = functools.wraps(
        function,
        #                               assigned=('__module__', '__name__', '__qualname__', '__doc__'),
        #                               updated=[],
    )(wrapper)

    #     wrapper.__signature__ = bound
    return wrapper


second_attempt = get_partial_formulary_function(blob, formulary.Bohm_diffusion)
# second_attempt()

functools.partial(<function Bohm_diffusion at 0x7fa103595940>, <Quantity 200. eV>, <Quantity 1. T>)

In [46]:
second_attempt?

Signature:       second_attempt(T_e: Unit("K"), B: Unit("T")) -> Unit("m2 / s")
Call signature:  second_attempt(*args, **kwargs)
Type:            partial
String form:     functools.partial(<function Bohm_diffusion at 0x7fa103595940>, <Quantity 200. eV>, <Quantity 1. T>)
File:            /mnt/hdd/Code/github/PlasmaPy/PlasmaPy/plasmapy/formulary/parameters.py
Docstring:      
Return the Bohm diffusion coefficient.

The Bohm diffusion coefficient was conjectured to follow Bohm model
of the diffusion of plasma across a magnetic field and describe the
diffusion of early fusion energy machines. The rate predicted by
Bohm diffusion is much higher than classical diffusion, and if there
were no exceptions, magnetically confined fusion would be impractical.

.. math::

    D_B = \frac{1}{16} \frac{k_B T}{e B}

where :math:`k_B` is the Boltzmann constant
and :math:`e` is the fundamental charge.

**Aliases:** `DB_`

Parameters
----------
T_e : `~astropy.units.Quantity`
    The electron temperature

# ... at scale

In [47]:
for name, function in inspect.getmembers(formulary):
    if not callable(function):
        continue
    partial_function = get_partial_formulary_function(blob, function)
    setattr(blob, function.__name__, partial_function)

In [48]:
blob.gyrofrequency("p+")

<Quantity 95788331.55943637 rad / s>

In [53]:
blob.Bohm_diffusion()

<Quantity 12.5 m2 / s>

# Flaw - field naming!

In [27]:
blob.Hall_parameter?

Signature:      
blob.Hall_parameter(
    n: Unit("1 / m3"),
    T: Unit("K"),
    B: Unit("T"),
    ion: plasmapy.particles.particle_class.Particle,
    particle: plasmapy.particles.particle_class.Particle,
    coulomb_log=None,
    V=None,
    coulomb_log_method='classical',
)
Call signature:  blob.Hall_parameter(*args, **kwargs)
Type:            partial
String form:     functools.partial(<function Hall_parameter at 0x7fa103585c10>, B=<Quantity 1. T>)
File:            /mnt/hdd/Code/github/PlasmaPy/PlasmaPy/plasmapy/formulary/parameters.py
Docstring:      
Calculate the ``particle`` Hall parameter for a plasma.

The Hall parameter for plasma species :math:`s` (``particle``) is given by:

.. math::

    β_{s} = \frac{Ω_{c s}}{ν_{s s^{\prime}}}

where :math:`Ω_{c s}` is the gyrofrequncy for plasma species :math:`s`
(``particle``) and :math:`ν_{s s^{\prime}}` is the collision frequency
between plasma species :math:`s` (``particle``) and species
:math:`s^{\prime}` (``ion``).

**Aliases:**

In [29]:
blob.Hall_parameter()

TypeError: missing a required argument: 'n'